# 05 · Final Evaluation on Hold-Out Period
### IT3091 Machine Learning · Group 2026-AI-42

Evaluation of candidate and tuned models on the chronologically held-out test period (newest 20% of contacts) to assess generalization performance and quantify the effect of time drift.


In [1]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import json, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, train_test_split

from src.config import *
from src.data.prepare import load_modelling_data
from src.evaluation.metrics import cv_eval, metric_row, scores_of, topk_metrics
from src.features.preprocessing import build_pipeline
from src.models.candidates import build_main_models, build_challenger_specs
from src.plotting import set_style, save_fig

set_style()
pd.set_option("display.max_columns", 60, "display.width", 170)
data = load_modelling_data()
X_train, y_train, X_test, y_test = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
X_all, y_all = data["X_all"], data["y_all"]
best_pipe = joblib.load(ARTIFACTS_DIR / "tuned_pipeline.joblib")
TUNED_NAME = json.loads((ARTIFACTS_DIR / "tuned_params.json").read_text())["name"]
models = build_main_models()

Loading local file: bank-additional-full.csv


Rows: raw 41,188 -> cleaned 41,176 | train 32,940 (rate 6.38%) | pilot 500 | test 7,736 (rate 32.41%)


---
## 1 · Hold-Out Evaluation: Main and Tuned Models


In [2]:
final_models = dict(models); final_models[TUNED_NAME] = best_pipe
rows, test_scores = [], {}
for name, p in final_models.items():
    m = clone(p).fit(X_train, y_train)
    s = scores_of(m, X_test)
    test_scores[name] = s
    r = metric_row(y_test, s); r["accuracy_at_0_5"] = accuracy_score(y_test, (s >= 0.5).astype(int))
    rows.append(dict(model=name, **r))
holdout = pd.DataFrame(rows).set_index("model").round(3)
display(holdout)
holdout.to_csv(TABLES_DIR / "holdout_results.csv")
np.savez(ARTIFACTS_DIR / "test_scores.npz", **{k: v for k, v in test_scores.items()})
print(f"Test base rate: {y_test.mean():.3%}  | budget = contact the best-scored {CALL_BUDGET:.0%} of clients")

,roc_auc,pr_auc,precision_at_budget,recall_at_budget,lift_at_budget,accuracy_at_0_5
model,,,,,,
Dummy (prior),0.500,0.324,0.304,0.187,0.937,0.676
Logistic regression,0.691,0.502,0.559,0.345,1.725,0.548
Random forest,0.687,0.507,0.590,0.364,1.821,0.701
Hist. gradient boosting,0.659,0.463,0.531,0.328,1.640,0.597
MLP (64-32),0.717,0.563,0.639,0.394,1.971,0.676
Tuned Logistic regression,0.689,0.499,0.557,0.344,1.719,0.681


Test base rate: 32.407%  | budget = contact the best-scored 20% of clients


### Observations
- Comparing candidate models against the dummy baseline confirms that models learn meaningful rank order above base rate.
- Because response rates drift between periods, cross-period comparison focuses on **lift** and **ROC-AUC** rather than raw PR-AUC.
- Global ROC-AUC captures both period-level baseline shifts and client-level ranking ability; the within-period discrimination is isolated in Notebook 06.

---
## 2 · Performance Comparison Across Model Families
Evaluating the full set of candidate models on the hold-out period for comparison against the selected pipeline.


In [3]:
zoo_specs = build_challenger_specs(y_train)
zoo_test = {}
for k, (m, sc) in zoo_specs.items():
    fitted = clone(build_pipeline(m, scale=sc)).fit(X_train, y_train)
    zoo_test[k] = metric_row(y_test, scores_of(fitted, X_test))
cols = ["roc_auc", "pr_auc", "precision_at_budget", "recall_at_budget", "lift_at_budget"]
zoo_holdout = pd.concat([holdout[cols], pd.DataFrame(zoo_test).T.round(3)]).sort_values("pr_auc", ascending=False)
display(zoo_holdout)
zoo_holdout.to_csv(TABLES_DIR / "holdout_results_with_challengers.csv")
print("The SVM row ranks by decision value, so its numbers are comparable on ranking metrics only.")

,roc_auc,pr_auc,precision_at_budget,recall_at_budget,lift_at_budget
MLP (64-32),0.717,0.563,0.639,0.394,1.971
SVM (RBF approx.),0.697,0.534,0.644,0.398,1.989
Random forest,0.687,0.507,0.590,0.364,1.821
Logistic regression,0.691,0.502,0.559,0.345,1.725
Tuned Logistic regression,0.689,0.499,0.557,0.344,1.719
XGBoost,0.673,0.491,0.559,0.345,1.723
Gradient boosting (sklearn),0.667,0.481,0.546,0.337,1.685
LightGBM,0.673,0.478,0.558,0.344,1.721
Hist. gradient boosting,0.659,0.463,0.531,0.328,1.640
Decision tree,0.645,0.418,0.548,0.338,1.689


The SVM row ranks by decision value, so its numbers are comparable on ranking metrics only.


---
## 3 · Consistency Across Validation Folds
Cross-referencing model rankings across stratified CV, forward-chaining CV, and hold-out test performance to verify stability and generalization.


In [4]:
cv_tab = pd.read_csv(TABLES_DIR / "model_comparison_stratified_cv.csv", index_col=0)
fc_tab = pd.read_csv(TABLES_DIR / "model_comparison_forward_chaining.csv", index_col=0)
print(f"1. Highest stratified-CV PR-AUC     : {cv_tab['pr_auc'].idxmax()} ({cv_tab['pr_auc'].max():.3f})")
print(f"2. Highest forward-chaining PR-AUC  : {fc_tab['pr_auc'].idxmax()} ({fc_tab['pr_auc'].max():.3f})")
print(f"3. Highest hold-out PR-AUC (main)   : {holdout['pr_auc'].idxmax()} ({holdout['pr_auc'].max():.3f})")
print(f"4. Highest hold-out PR-AUC (all)    : {zoo_holdout['pr_auc'].idxmax()} ({zoo_holdout['pr_auc'].max():.3f})")
print(f"5. Highest hold-out lift (main)     : {holdout['lift_at_budget'].idxmax()} ({holdout['lift_at_budget'].max():.2f}x)")

1. Highest stratified-CV PR-AUC     : Hist. gradient boosting (0.210)
2. Highest forward-chaining PR-AUC  : Logistic regression (0.098)
3. Highest hold-out PR-AUC (main)   : MLP (64-32) (0.563)
4. Highest hold-out PR-AUC (all)    : MLP (64-32) (0.563)
5. Highest hold-out lift (main)     : MLP (64-32) (1.97x)


---
## 4 · Robustness and Split Methodology

### 4.1 Cross-Validation vs Forward-Chaining
Comparing shuffled 5-fold cross-validation with time-series forward-chaining folds on the dataset to estimate deployment stability under temporal drift.


In [5]:
rand_auc = cv_eval(clone(best_pipe), X_train, y_train, 5)["roc_auc"]
ts_rows = []
for i, (tr, va) in enumerate(TimeSeriesSplit(n_splits=5).split(X_all)):
    m = clone(best_pipe).fit(X_all.iloc[tr], y_all[tr]); s = scores_of(m, X_all.iloc[va])
    ts_rows.append(dict(fold=i + 1, train_rows=len(tr), val_rows=len(va), val_rate=y_all[va].mean(), roc_auc=roc_auc_score(y_all[va], s), pr_auc=average_precision_score(y_all[va], s)))
ts = pd.DataFrame(ts_rows).set_index("fold").round(3)
display(ts)
print(f"ROC-AUC  random 5-fold CV: {rand_auc.mean():.3f} +/- {rand_auc.std():.3f}  |  forward-chaining: {ts['roc_auc'].mean():.3f} +/- {ts['roc_auc'].std():.3f}")
print(f"Chronological hold-out ROC-AUC (section 1): {roc_auc_score(y_test, test_scores[TUNED_NAME]):.3f}")

,train_rows,val_rows,val_rate,roc_auc,pr_auc
fold,,,,,
1,6866,6862,0.046,0.555,0.055
2,13728,6862,0.065,0.540,0.074
3,20590,6862,0.053,0.446,0.049
4,27452,6862,0.127,0.663,0.233
5,34314,6862,0.355,0.676,0.511


ROC-AUC  random 5-fold CV: 0.667 +/- 0.021  |  forward-chaining: 0.576 +/- 0.095
Chronological hold-out ROC-AUC (section 1): 0.689


### 4.2 Quantifying Split Optimism
Evaluating performance differences between standard stratified random 80/20 splits and the chronological hold-out to assess optimism bias in non-temporal validation.


In [6]:
rows = []
for sd in range(3):
    Xa, Xb, ya, yb = train_test_split(X_all, y_all, test_size=TEST_FRACTION, stratify=y_all, random_state=SEED + sd)
    r = metric_row(yb, scores_of(clone(best_pipe).fit(Xa, ya), Xb)); r["base_rate"] = yb.mean(); r["split"] = f"stratified random #{sd + 1}"; rows.append(r)
rand_split = pd.DataFrame(rows).set_index("split")
chrono_row = pd.Series({**metric_row(y_test, test_scores[TUNED_NAME]), "base_rate": y_test.mean()}, name="chronological hold-out")
split_table = pd.concat([rand_split, chrono_row.to_frame().T])
display(split_table.round(3))
split_table.to_csv(TABLES_DIR / "split_optimism.csv")
opt = rand_split.mean() - chrono_row
print("Optimism of a stratified random split (mean of 3 splits minus chronological hold-out):")
print(opt[["roc_auc", "lift_at_budget"]].round(3).to_string())
print("PR-AUC is not compared here: it depends on the base rate, which differs between a random sample and the later period.")

,roc_auc,pr_auc,precision_at_budget,recall_at_budget,lift_at_budget,base_rate
stratified random #1,0.765,0.406,0.323,0.573,2.867,0.113
stratified random #2,0.759,0.403,0.312,0.554,2.770,0.113
stratified random #3,0.763,0.411,0.315,0.558,2.791,0.113
chronological hold-out,0.689,0.499,0.557,0.344,1.719,0.324


Optimism of a stratified random split (mean of 3 splits minus chronological hold-out):
roc_auc           0.073
lift_at_budget    1.090
PR-AUC is not compared here: it depends on the base rate, which differs between a random sample and the later period.


### Summary of Split Impact
Random stratified splitting mixes records across all campaign months, allowing models to exploit period-specific artifacts and inflating ROC-AUC and lift. Chronological hold-out provides an honest estimate of real-world future performance.
